In [4]:
# diagnose environment inside the notebook
import sys, pkgutil
print("Python executable:", sys.executable)
print("bitsandbytes available?", pkgutil.find_loader("bitsandbytes"))
import bitsandbytes as bnb
print("bitsandbytes version", bnb.__version__)

Python executable: /home/asier/miniconda3/envs/medgemma2/bin/python
bitsandbytes available? <_frozen_importlib_external.SourceFileLoader object at 0x7f9ae0767460>
bitsandbytes version 0.49.2


In [5]:
# Make sure to install the required libraries first:
# pip install accelerate bitsandbytes
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from PIL import Image
import requests
import torch

#from huggingface_hub import login
#login(token="YOUR_TOKEN_HERE")

model_id = "google/medgemma-1.5-4b-it"

# 4-bit NF4 quantization — cuts VRAM from ~8 GB (bf16) to ~2.5 GB
# and significantly increases token throughput on consumer GPUs.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # NF4 is recommended for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16, # compute in bf16 for accuracy
    bnb_4bit_use_double_quant=True,        # extra ~0.4 bits/param saving
)



In [6]:


model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",         # Accelerate places layers optimally
    low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(model_id)

# Image attribution: Stillwaterising, CC0, via Wikimedia Commons
image_url = "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png"
image = Image.open(requests.get(image_url, headers={"User-Agent": "example"}, stream=True).raw)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Describe this X-ray"}
        ]
    }
]


inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device)

input_len = inputs["input_ids"].shape[-1]



with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=2000, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


This is a chest X-ray image. Here's a description of what I can see:

*   **Bones:** The ribs, clavicles (collarbones), and the upper portions of the lungs are visible.
*   **Lungs:** The lungs appear relatively clear, with no obvious large areas of consolidation (like pneumonia) or significant opacities.
*   **Heart:** The heart silhouette is visible, though the size and shape are not easily determined without more information.
*   **Mediastinum:** The mediastinum (the space between the lungs containing the heart, major blood vessels, and trachea) appears normal.
*   **Diaphragm:** The diaphragm is visible at the bottom of the image.

Overall, the image appears to be a normal chest X-ray. However, a definitive diagnosis requires a trained radiologist to interpret the image in the context of the patient's clinical history and other relevant information.



In [7]:
# ---------------------------------------------------------------------------
# Save the 4-bit quantized model to disk so Flask can load it without
# re-downloading from HuggingFace every restart.
# Point the Flask server at this directory with:
#   export LOCAL_MODEL_DIR=/path/to/quant-medgemma
#   python api_stable.py
# ---------------------------------------------------------------------------
SAVE_DIR = "./quant-medgemma"

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print(f"Saved quantized model to {SAVE_DIR}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved quantized model to ./quant-medgemma
